# 0.0 - Imports

## 0.1 - Importa as bibliotecas

In [87]:
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

## 0.2 - Carrega as credenciais do .env

In [88]:
load_dotenv()

True

## 0.3 - Conexão utilizando o .env

In [89]:
user = os.getenv('DB_USER')
password = os.getenv('DB_PASS')
host = os.getenv('DB_HOST')
dataset = os.getenv('DB_NAME')

url = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{dataset}")

## 0.4 - Teste de Conexão

In [90]:
with url.connect() as conn:
    result = conn.execute(text("SELECT VERSION();"))
    print(result.fetchone())

('8.0.31-google',)


# 1.0 - Execução das consultas

In [91]:
#Transforma as query em texto para pandas
query_input = text("SELECT * FROM data_product_sales dps")

In [92]:
#Carrega no pandas as duas tabelas
df_raw = pd.read_sql_query(query_input, con=url)

In [93]:
#Verificação dos tipos do DataFrame
print(df_raw.dtypes)

STORE_CODE          str
PRODUCT_CODE      int64
DATE             object
SALES_VALUE     float64
SALES_QTY       float64
dtype: object


In [94]:
#Verificação de NaN no dataset
df_raw.isna().sum()

STORE_CODE      0
PRODUCT_CODE    0
DATE            0
SALES_VALUE     0
SALES_QTY       0
dtype: int64

In [95]:
#Verifica se datasets estão de acordo com exemplos
print(df_raw.head(5))

  STORE_CODE  PRODUCT_CODE        DATE  SALES_VALUE  SALES_QTY
0          1            18  2019-01-01        708.5       65.0
1          1            18  2019-01-02       1297.1      119.0
2          1            18  2019-01-03       1144.5      105.0
3          1            18  2019-01-04       1090.0      100.0
4          1            18  2019-01-05        893.8       82.0


# 2.0 - Construção da Função

### 2.1 - Construção da função

In [96]:
def retrieve_data(df, product_code, store_code, date):
    """
    Filtra o DataFrame de vendas por produto, loja e intervalo de datas,
    exporta o resultado em .csv e retorna o DataFrame filtrado.
    
    Responde às perguntas:
    "Como otimizar as consultas independente dos filtros ?"

    Parâmetros
    ----------
    df : pd.DataFrame
        DataFrame bruto gerado pela consulta ao banco MySQL (df_raw_sales).
    
    product_code : int, list ou None
        Código do produto. Aceita valor único, lista de códigos ou None
        (retorna DataFrame vazio).
    
    store_code : int, list ou None
        Código da loja. Aceita valor único, lista de códigos ou None
        (retorna DataFrame vazio).
    
    date : list of str
        Intervalo de datas no formato ISO. Sempre dois elementos:
        Exemplo 1: ['2019-01-01', '2019-01-31'] → range de datas.
        Exemplo 2: ['2019-01-15', '2019-01-15'] → data única para fitrar apenas um dia.
    
    Retorna
    -------
    df_final : pd.DataFrame
        DataFrame filtrado com as colunas na ordem:
        STORE_CODE, PRODUCT_CODE, DATE, SALES_VALUE, SALES_QTY.
    """
    
    # 1. Validação de entrada
    
    if product_code is None or store_code is None or date is None:
        return pd.DataFrame(columns=['STORE_CODE', 'PRODUCT_CODE', 'DATE', 'SALES_VALUE', 'SALES_QTY'])
    
    # 1.1 Cria uma cópia do Dataframe Original para Manipulação
    df1 = df.copy()
    
    # 2. Ajusta todos os tipos de dados do dataset (transformação do )
    df1['STORE_CODE'] = df1["STORE_CODE"].astype(int)
    df1['DATE'] = pd.to_datetime(df1['DATE'])
    
    # 3. Lista as variaveis de filtro para o dataset
    flt_product_code = np.atleast_1d(product_code)
    flt_store_code = np.atleast_1d(store_code)
    flt_date_start = pd.to_datetime(date[0])
    flt_date_end = pd.to_datetime(date[1])
    
    # 4. Cria o filtro para passar no dataset
    filtro = (
        df1['STORE_CODE'].isin(flt_store_code) &
        df1['PRODUCT_CODE'].isin(flt_product_code) &
        df1['DATE'].between(flt_date_start, flt_date_end)
    )
    df_filtro = df1.loc[filtro]
    
    # 5. Organiza as colunas do dataset para exportação final (em caso de personalização de exportação)
    cols_ordem = ['STORE_CODE', 'PRODUCT_CODE', 'DATE', 'SALES_VALUE', 'SALES_QTY']
    df_final = df_filtro[cols_ordem].reset_index(drop=True)
    
    # 6. Exporta o df_final em .csv para utilização em outros SAAS
    output_dir  = os.path.join(os.getcwd(), "..", "02_retrieve_data", "data")
    output_path = os.path.join(output_dir, "resultado_filtrado.csv")
    
    os.makedirs("02_retrieve_data/data", exist_ok=True)
    df_final.to_csv(output_path, index=False, sep=";", encoding="utf-8-sig")
    
    return df_final

### 2.2 - Teste da função final

#### 2.2.1 - Função Completa

In [86]:
retrieve_data(df_raw, 18, 1, ['2019-01-01', '2019-01-31'])

,STORE_CODE,PRODUCT_CODE,DATE,SALES_VALUE,SALES_QTY
0,1,18,2019-01-01,708.5,65.0
1,1,18,2019-01-02,1297.1,119.0
2,1,18,2019-01-03,1144.5,105.0
3,1,18,2019-01-04,1090.0,100.0
4,1,18,2019-01-05,893.8,82.0
5,1,18,2019-01-06,741.2,68.0
6,1,18,2019-01-07,654.0,60.0
7,1,18,2019-01-08,741.2,68.0
8,1,18,2019-01-09,1373.4,126.0
9,1,18,2019-01-10,1068.2,98.0


#### 2.2.2 - Função sem Product Code

#### 2.2.3 - Função sem Store Code

#### 2.2.4 - Função sem Data Inicio

#### 2.2.2 - Função sem Data Final